# SPFC Evaluation Bench - Kaggle T2I Only

Kaggle runbook for SPFC vs Rectified-CFG++ vs base SD3 Medium on the 100-prompt T2I-CompBench subset only. Every generation/evaluation run is isolated in its own cell and streams progress while it runs.

In [ ]:
GITHUB_REPO_URL = 'https://github.com/Soobiwan/aim-flow.git'
SEED = 13
RUN_ROOT = '/kaggle/working/spfc_eval_seed13/runs'
REPORT_DIR = '/kaggle/working/spfc_eval_seed13/reports'
EVAL_DIR = f'{REPORT_DIR}/eval'
T2I_MANIFEST = '/kaggle/working/aim-flow/configs/t2i_compbench_100_seed13.json'
T2I_DECOMP = '/kaggle/working/aim-flow/configs/t2i_compbench_100_seed13_spfc.json'
T2I_DATASET_ROOT = '/kaggle/working/aim-flow/external/T2I-CompBench/examples/dataset'
EXECUTE_T2I_OFFICIAL = True
T2I_EVAL_CATEGORIES = 'color shape texture spatial'
QUALITATIVE_MANIFEST = T2I_MANIFEST

In [ ]:
%cd /kaggle/working
!rm -rf /kaggle/working/aim-flow
!git clone {GITHUB_REPO_URL} /kaggle/working/aim-flow
%cd /kaggle/working/aim-flow

In [ ]:
# Core package setup. This cell only changes Torch when the active GPU requires it.
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'datasets'], check=True)

import torch

print('python:', sys.version)
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    device_name = torch.cuda.get_device_name(0)
    capability = torch.cuda.get_device_capability(0)
    required_arch = f'sm_{capability[0]}{capability[1]}'
    supported_arches = set(torch.cuda.get_arch_list())
    print('gpu:', device_name, required_arch)
    print('torch cuda arches:', sorted(supported_arches))
    if supported_arches and required_arch not in supported_arches:
        subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torch', 'torchvision', 'torchaudio'], check=True)
        subprocess.run([
            sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--force-reinstall',
            'torch==2.4.1+cu118', 'torchvision==0.19.1+cu118',
            '--index-url', 'https://download.pytorch.org/whl/cu118',
        ], check=True)
        raise RuntimeError('Installed a CUDA 11.8 Torch build for this GPU. Restart the Kaggle kernel, then rerun from this cell.')

In [ ]:
# Official T2I-CompBench evaluator dependencies.
# Detectron2 is built from current source because the old pinned T2I-CompBench commit does not build reliably on Kaggle Python 3.12 / Torch 2.4.
import os
import subprocess
import sys

base_packages = [
    'timm==0.4.12',
    'fairscale==0.4.4',
    'ruamel.yaml',
    'opencv-python',
    'yacs',
    'pycocotools',
    'spacy',
    'ftfy',
    'regex',
    'ninja',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', *base_packages], check=True)
subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'], check=True)

try:
    import detectron2  # noqa: F401
    print('detectron2 already importable')
except Exception:
    import torch

    build_env = os.environ.copy()
    build_env.setdefault('MAX_JOBS', '2')
    if torch.cuda.is_available():
        major, minor = torch.cuda.get_device_capability(0)
        build_env.setdefault('TORCH_CUDA_ARCH_LIST', f'{major}.{minor}')
    print('building detectron2 from source with MAX_JOBS=', build_env.get('MAX_JOBS'))
    subprocess.run(
        [
            sys.executable,
            '-m',
            'pip',
            'install',
            '--no-build-isolation',
            '--no-cache-dir',
            'git+https://github.com/facebookresearch/detectron2.git',
        ],
        env=build_env,
        check=True,
        timeout=3600,
    )
    import detectron2
    print('detectron2 installed:', getattr(detectron2, '__version__', 'unknown'))

In [ ]:
import shutil
import subprocess
from pathlib import Path

import requests
from tqdm.auto import tqdm

t2i_repo = Path('/kaggle/working/aim-flow/external/T2I-CompBench')
if not t2i_repo.exists():
    t2i_repo.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', 'https://github.com/Karine-Huang/T2I-CompBench.git', str(t2i_repo)], check=True)
subprocess.run(['git', '-C', str(t2i_repo), 'fetch', 'origin', '1b7094991a57f3c22abdd4f6e8ba6c1a15517073'], check=True)
subprocess.run(['git', '-C', str(t2i_repo), 'checkout', '1b7094991a57f3c22abdd4f6e8ba6c1a15517073'], check=True)

weight_dir = t2i_repo / 'UniDet_eval' / 'experts' / 'expert_weights'
weight_dir.mkdir(parents=True, exist_ok=True)
weight_path = weight_dir / 'Unified_learned_OCIM_RS200_6x+2x.pth'
weight_url = 'https://huggingface.co/shikunl/prismer/resolve/main/expert_weights/Unified_learned_OCIM_RS200_6x%2B2x.pth'
if not weight_path.exists() or weight_path.stat().st_size < 100_000_000:
    tmp_path = weight_path.with_suffix('.pth.part')
    with requests.get(weight_url, stream=True, timeout=(30, 120)) as response:
        response.raise_for_status()
        total = int(response.headers.get('content-length') or 0)
        with tmp_path.open('wb') as f, tqdm(total=total, unit='B', unit_scale=True, desc='UniDet RS200') as bar:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
                    bar.update(len(chunk))
    shutil.move(str(tmp_path), str(weight_path))
if weight_path.stat().st_size < 100_000_000:
    raise RuntimeError(f'UniDet checkpoint looks incomplete: {weight_path} ({weight_path.stat().st_size} bytes)')
print(f'Official T2I evaluator weights are ready: {weight_path} ({weight_path.stat().st_size / 1e9:.2f} GB)')

In [ ]:
# Official evaluator smoke test: load BLIP-VQA and UniDet/Detectron2 before running the full eval.
import subprocess
import sys
import textwrap
from pathlib import Path

T2I_REPO = Path('/kaggle/working/aim-flow/external/T2I-CompBench')

def run_smoke(label, cwd, code, timeout=1800):
    print(f'--- {label} ---')
    result = subprocess.run(
        [sys.executable, '-u', '-c', code],
        cwd=str(cwd),
        text=True,
        timeout=timeout,
    )
    if result.returncode != 0:
        raise RuntimeError(f'{label} failed with exit code {result.returncode}')

blip_code = r'''
import gc
import torch
from models.blip_vqa import blip_vqa

print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
model = blip_vqa(
    pretrained='https://storage.googleapis.com/sfr-vision-language-research/BLIP/models/model_base_vqa_capfilt_large.pth',
    image_size=480,
    vit='base',
    vit_grad_ckpt=False,
    vit_ckpt_layer=0,
)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = model.to(device)
print('BLIP loaded on', device)
del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
'''

unidet_code = r'''
import gc
import torch
import detectron2
from experts.model_bank import load_expert_model

print('detectron2', getattr(detectron2, '__version__', 'unknown'))
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
model, transform = load_expert_model(task='obj_detection', ckpt='RS200')
print('UniDet loaded:', type(model).__name__, 'transform:', transform)
del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
'''

run_smoke('BLIP-VQA model load', T2I_REPO / 'BLIPvqa_eval', blip_code)
run_smoke('UniDet / Detectron2 model load', T2I_REPO / 'UniDet_eval', unidet_code)
print('Official evaluator smoke test passed.')

In [ ]:
import os
from IPython.display import Markdown, display

MODEL_ID = 'stabilityai/stable-diffusion-3-medium-diffusers'
HF_SECRET_NAMES = ('Huggingface', 'HF_TOKEN', 'HUGGINGFACE_TOKEN')

if not (os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_TOKEN')):
    try:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        for secret_name in HF_SECRET_NAMES:
            try:
                token = secrets.get_secret(secret_name)
            except Exception:
                token = None
            if token:
                os.environ['HF_TOKEN'] = token
                break
    except Exception:
        pass

token = os.environ.get('HF_TOKEN') or os.environ.get('HUGGINGFACE_TOKEN')
if not token:
    raise RuntimeError(
        'Missing Hugging Face token. In Kaggle, add a secret named Huggingface or HF_TOKEN, '
        'turn it on for this notebook, and make sure that Hugging Face account has accepted the SD3 Medium license.'
    )

try:
    from huggingface_hub import HfApi
    HfApi().model_info(MODEL_ID, token=token)
except Exception as exc:
    raise RuntimeError(
        f'HF_TOKEN is set, but access check for {MODEL_ID} failed. '
        'Confirm the Kaggle secret is enabled and the token account has accepted the gated model license.'
    ) from exc

display(Markdown('Hugging Face token is configured and can access SD3 Medium.'))


In [ ]:
import json
import subprocess
import time
from pathlib import Path
from IPython.display import Image as DisplayImage, Markdown, display

EST_SEC_PER_PROMPT = {
    'spfc_generation': 240,
    'rectified_cfgpp_generation': 90,
    'base_generation': 45,
    't2i_official_eval': 8,
    't2i_stage_only': 0.05,
}


def manifest_count(path, default=100):
    path = Path(path)
    if not path.exists():
        return default
    return len(json.loads(path.read_text(encoding='utf-8'))['samples'])


def fmt_seconds(seconds):
    seconds = int(seconds)
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f'{h}h {m}m {s}s' if h else f'{m}m {s}s'


def timed_run(label, command, estimated_seconds, timeout_seconds=None):
    timeout_note = f'\nTimeout: **{fmt_seconds(timeout_seconds)}**' if timeout_seconds else ''
    display(Markdown(f'### {label}\nEstimated time: **{fmt_seconds(estimated_seconds)}**{timeout_note}'))
    start = time.perf_counter()
    try:
        result = subprocess.run(command, shell=True, text=True, timeout=timeout_seconds)
    except subprocess.TimeoutExpired as exc:
        elapsed = time.perf_counter() - start
        raise TimeoutError(f'{label} timed out after {fmt_seconds(elapsed)}: {command}') from exc
    elapsed = time.perf_counter() - start
    display(Markdown(f'Finished **{label}** in **{fmt_seconds(elapsed)}**.'))
    if result.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {result.returncode}: {command}')


def t2i_category_flag():
    return f'--t2i-categories {T2I_EVAL_CATEGORIES}' if T2I_EVAL_CATEGORIES else ''


def show_scores(path):
    path = Path(path)
    if path.exists():
        data = json.loads(path.read_text(encoding='utf-8'))
        display(Markdown('```json\n' + json.dumps(data.get('scores', data), indent=2) + '\n```'))


def show_markdown(path):
    path = Path(path)
    if path.exists():
        display(Markdown(path.read_text(encoding='utf-8')))


def show_image(path):
    path = Path(path)
    if path.exists():
        display(DisplayImage(filename=str(path)))

In [ ]:
# timed_run('Prepare T2I-CompBench 100-prompt manifest and SPFC template', f'python scripts/bench_prepare_prompts.py --benchmark t2i_compbench --seed {SEED} --t2i-subset-size 100 --t2i-dataset-root {T2I_DATASET_ROOT} --write-decomposition-template', 5)

Replace the generated decomposition templates with real LLM/manual SPFC decompositions before generation.

In [ ]:
# timed_run('Validate T2I SPFC decompositions', f'python scripts/bench_validate_decompositions.py --manifest {T2I_MANIFEST} --decompositions {T2I_DECOMP}', 2)

In [ ]:
# One-image generation smoke test. This catches SD3 weight-loading/runtime problems before the 100-image run.
SMOKE_DIR = Path('/kaggle/working/spfc_eval_seed13/smoke')
SMOKE_DIR.mkdir(parents=True, exist_ok=True)
SMOKE_MANIFEST = SMOKE_DIR / 't2i_smoke_manifest.json'
SMOKE_DECOMP = SMOKE_DIR / 't2i_smoke_spfc.json'
SMOKE_RUN_ROOT = SMOKE_DIR / 'runs'

manifest_data = json.loads(Path(T2I_MANIFEST).read_text(encoding='utf-8'))
decomp_data = json.loads(Path(T2I_DECOMP).read_text(encoding='utf-8'))
manifest_data['samples'] = manifest_data['samples'][:1]
first_id = manifest_data['samples'][0]['id']
decomp_data['items'] = [item for item in decomp_data['items'] if item['id'] == first_id]
SMOKE_MANIFEST.write_text(json.dumps(manifest_data, indent=2), encoding='utf-8')
SMOKE_DECOMP.write_text(json.dumps(decomp_data, indent=2), encoding='utf-8')

timed_run(
    'SPFC one-image smoke generation',
    f'python -u scripts/bench_generate.py --manifest {SMOKE_MANIFEST} --decompositions {SMOKE_DECOMP} --run-root {SMOKE_RUN_ROOT} --methods spfc --seed {SEED} --config configs/sd3_medium_kaggle.yaml',
    EST_SEC_PER_PROMPT['spfc_generation'],
    timeout_seconds=1800,
)

In [ ]:
N = manifest_count(T2I_MANIFEST)
timed_run('SPFC T2I-CompBench generation', f'python -u scripts/bench_generate.py --manifest {T2I_MANIFEST} --decompositions {T2I_DECOMP} --run-root {RUN_ROOT} --methods spfc --seed {SEED} --config configs/sd3_medium_kaggle.yaml', N * EST_SEC_PER_PROMPT['spfc_generation'])

In [ ]:
N = manifest_count(T2I_MANIFEST)
timed_run('Rectified-CFG++ T2I-CompBench generation', f'python -u scripts/bench_generate.py --manifest {T2I_MANIFEST} --run-root {RUN_ROOT} --methods rectified_cfgpp --seed {SEED} --config configs/sd3_medium_kaggle.yaml', N * EST_SEC_PER_PROMPT['rectified_cfgpp_generation'])

In [ ]:
N = manifest_count(T2I_MANIFEST)
timed_run('Base SD3 T2I-CompBench generation', f'python -u scripts/bench_generate.py --manifest {T2I_MANIFEST} --run-root {RUN_ROOT} --methods base --seed {SEED} --config configs/sd3_medium_kaggle.yaml', N * EST_SEC_PER_PROMPT['base_generation'])

In [ ]:
score_path = Path(EVAL_DIR) / 't2i_compbench_scores.json'
if score_path.exists():
    score_path.unlink()

In [ ]:
flag = '--execute-official' if EXECUTE_T2I_OFFICIAL else ''
N = manifest_count(T2I_MANIFEST)
estimate = N * (EST_SEC_PER_PROMPT['t2i_official_eval'] if EXECUTE_T2I_OFFICIAL else EST_SEC_PER_PROMPT['t2i_stage_only'])
timed_run('SPFC T2I-CompBench evaluation', f'python -u scripts/bench_evaluate.py --benchmark t2i_compbench --manifest {T2I_MANIFEST} --run-root {RUN_ROOT} --output-dir {EVAL_DIR} --methods spfc --append {flag} {t2i_category_flag()}', estimate)
show_scores(Path(EVAL_DIR) / 't2i_compbench_scores.json')

In [ ]:
flag = '--execute-official' if EXECUTE_T2I_OFFICIAL else ''
N = manifest_count(T2I_MANIFEST)
estimate = N * (EST_SEC_PER_PROMPT['t2i_official_eval'] if EXECUTE_T2I_OFFICIAL else EST_SEC_PER_PROMPT['t2i_stage_only'])
timed_run('Rectified-CFG++ T2I-CompBench evaluation', f'python -u scripts/bench_evaluate.py --benchmark t2i_compbench --manifest {T2I_MANIFEST} --run-root {RUN_ROOT} --output-dir {EVAL_DIR} --methods rectified_cfgpp --append {flag} {t2i_category_flag()}', estimate)
show_scores(Path(EVAL_DIR) / 't2i_compbench_scores.json')

In [ ]:
flag = '--execute-official' if EXECUTE_T2I_OFFICIAL else ''
N = manifest_count(T2I_MANIFEST)
estimate = N * (EST_SEC_PER_PROMPT['t2i_official_eval'] if EXECUTE_T2I_OFFICIAL else EST_SEC_PER_PROMPT['t2i_stage_only'])
timed_run('Base SD3 T2I-CompBench evaluation', f'python -u scripts/bench_evaluate.py --benchmark t2i_compbench --manifest {T2I_MANIFEST} --run-root {RUN_ROOT} --output-dir {EVAL_DIR} --methods base --append {flag} {t2i_category_flag()}', estimate)
show_scores(Path(EVAL_DIR) / 't2i_compbench_scores.json')

In [ ]:
T2I_SCORES = f'{EVAL_DIR}/t2i_compbench_scores.json'
QUAL_GRID = f'{REPORT_DIR}/qualitative_grid.png'
timed_run('Build T2I table and qualitative grid', f'python scripts/bench_report.py --t2i-scores {T2I_SCORES} --run-root {RUN_ROOT} --output-dir {REPORT_DIR} --qualitative-manifest {QUALITATIVE_MANIFEST} --qualitative-output {QUAL_GRID}', 10)
show_markdown(Path(REPORT_DIR) / 't2i_compbench_table.md')
show_image(QUAL_GRID)